# Kuvantunnistus omilla kuvilla - CNN

In [ ]:
import tensorflow as tf
import numpy as np
from numpy.f2py.crackfortran import verbose
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, RandomFlip, RandomRotation, RandomZoom
import random

batch_size = 32
img_size = (224, 224)
split_ratio = [0.7, 0.15, 0.15] 

# Ladataan datasetti
dataset = tf.keras.preprocessing.image_dataset_from_directory(
    "../datasets/kuvantunnistus",
    image_size=img_size,
    batch_size=batch_size,
    label_mode="categorical",
    shuffle=True,
)

class_names = dataset.class_names

# Normalisoidaan kuvat
dataset = dataset.map(lambda x, y: (x / 255.0, y))

dataset_size = len(dataset)


train_size = int(split_ratio[0] * dataset_size)
val_size = int(split_ratio[1] * dataset_size)

train_dataset = dataset.take(train_size)
validation_dataset = dataset.skip(train_size).take(val_size)
test_dataset = dataset.skip(train_size + val_size)


# Tarkistetaan datasetin koko
print(f"Train batches: {len(train_dataset)}, Validation batches: {len(validation_dataset)}, Test batches: {len(test_dataset)}")


In [ ]:
data_augmentation = Sequential([
    RandomFlip("horizontal"),
    RandomRotation(0.1),
    RandomZoom(0.2),
])

model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)),
    data_augmentation,
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(6, activation='softmax')
])

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

history = model.fit(train_dataset, validation_data=validation_dataset, epochs=10, verbose=0)

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history["loss"], label="Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.legend()
plt.grid()
plt.show()

In [ ]:
score = model.evaluate(test_dataset, verbose=0)
print('Test loss:', score[0])
print('Test accuracy:', score[1])

In [ ]:
# Haetaan testidatasetista yksi batch
test_images, test_labels = next(iter(test_dataset.take(1)))

# Ennustetaan mallilla
predictions = model.predict(test_images)

# Muunnetaan one-hot-enkoodatut labelit kokonaisluvuiksi
true_labels = np.argmax(test_labels, axis=1)
predicted_labels = np.argmax(predictions, axis=1)

def plot_images(model):
    y_pred = []
    x_images = []
    y_true = []

    for images, labels in test_dataset:
        y_pred.extend(model.predict(images, batch_size=32))
        x_images.extend(images.numpy())
        y_true.extend(labels.numpy())

    y_pred = np.array(y_pred)
    y_true = np.array(y_true)
    x_images = np.array(x_images)

    num_samples = 10
    random_indices = random.sample(range(len(x_images)), num_samples)

    for i in random_indices:
        plt.figure(figsize=(10, 4))

        # Display image
        plt.subplot(1, 2, 1)
        plt.imshow((x_images[i] * 255).astype("uint8"))
        true_class = class_names[np.argmax(y_true[i])]
        plt.title(f"True: {true_class}")
        plt.axis('off')

        plt.subplot(1, 2, 2)
        pred_prob = y_pred[i]
        predicted_class = class_names[np.argmax(pred_prob)]

        colors = ['lightgreen' if (class_names[idx] == true_class) else 'skyblue'
                  for idx in range(len(class_names))]

        plt.barh(class_names, pred_prob, color=colors)
        plt.xlabel('Probability')
        plt.title(f"Predicted: {predicted_class}")
        plt.xlim([0, 1])

        if predicted_class != true_class:
            plt.text(0.5, -0.5, "Incorrect", color='red',
                    ha='center', transform=plt.gca().transAxes)
        else:
            plt.text(0.5, -0.5, "Correct", color='green',
                    ha='center', transform=plt.gca().transAxes)

        plt.tight_layout()
        plt.show()

plot_images(model)

# Kuvantunnistus omilla kuvilla - VGG16 Assisted
Luodaan malli VGG16:n avulla, joka on esikoulutettu ImageNet-datasetillä. Mallista jätetään pois viimeinen kerros, koska sen halutaan sopivan omaan datasettiin. Mallin kerrokset jäädytetään, jotta niitä ei kouluteta uudelleen. Mallin päälle lisätään oma toteutus.

In [ ]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Flatten, Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.regularizers import l2

# Ladataan VGG16-malli ilman viimeistä kerrosta

vgg16_base = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
vgg16_base.trainable = False

Luodaan omat kerrokset VGG16-mallin päälle. Mallin rakenne on seuraava:
- GlobalAveragePooling2D
- Dense(256, activation='relu')
- Dense(128, activation='relu', kernel_regularizer=l2(0.01))
- Dense(128, activation='relu')
- Dropout(0.2)
- Dense(6, activation='softmax')

In [ ]:
x = vgg16_base.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dense(128, activation='relu', kernel_regularizer=l2(0.01))(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.2)(x)
predictions = Dense(6, activation='softmax')(x)

Rakennetaan malli VGG16:n ja omien kerrosten avulla. Mallin optimointimenetelmänä käytetään Adamia, häviönä categorical_crossentropy ja mittarina tarkkuus.
Koulutetaan mallia 10 epookkia, eräkoolla 32.

In [ ]:
model2 = Model(inputs=vgg16_base.input, outputs=predictions)

model2.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

history2 = model2.fit(train_dataset, validation_data=validation_dataset, epochs=10, verbose=0)

In [ ]:
plt.plot(history2.history["loss"], label="Loss")
plt.plot(history2.history["val_loss"], label="Validation Loss")
plt.legend()
plt.grid()
plt.show()

In [ ]:
model2_test_score = model2.evaluate(test_dataset, verbose=0)
print('Test loss:', model2_test_score[0])
print('Test accuracy:', model2_test_score[1])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

test_loss, test_acc = model.evaluate(test_dataset)
print(f"\nTest Set Accuracy: {test_acc*100:.2f}%\n")

plot_images(model2)